# How do you score a match between two lists of numbers?

Your music app just played you a song you love, by a band you've never heard of. Somewhere in a data center, two lists of numbers got compared, and the comparison said: same taste.

We're going to build that comparison by hand, in NumPy, and watch it work.

Rule for the whole notebook: only `numpy` and the standard library. No hidden magic, no plotting library standing in for understanding. If we want a picture, we print numbers with labels.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
print("ready")

## A taste is an arrow

Strip a taste down to its bones: each direction is a quality, and the length along that direction is how hard the song leans that way. Two dimensions, so we can hold the whole thing in our head.

Below is one amber probe (that's you) and four candidate songs, A through D. They're just pairs of numbers.

Before you run the next cell: which candidate do you think will score highest against the probe `[3.0, 0.0]`? Do you expect any score to go negative?

In [ ]:
probe = np.array([3.0, 0.0])

candidates = {
    "A": np.array([3.0, 1.0]),
    "B": np.array([1.0, 3.0]),
    "C": np.array([-2.0, 2.0]),
    "D": np.array([-3.0, -1.0]),
}

for name, vec in candidates.items():
    print(name, vec)

## The rule: multiply entry by entry, then add

That's the whole recipe for a match score. No division, no square roots, nothing fancy. Multiply the matched entries, add the products, done.

Let's write it by hand first, so there's no doubt what's happening under `np.dot` later.

In [ ]:
def dot_by_hand(u, v):
    return sum(x * y for x, y in zip(u, v))

scores = {name: dot_by_hand(vec, probe) for name, vec in candidates.items()}
for name, score in scores.items():
    print(name, score)

# our hand-rolled version and NumPy's built-in must agree
for name, vec in candidates.items():
    assert dot_by_hand(vec, probe) == np.dot(vec, probe)

A comes out on top, and nothing went negative yet. Hold that thought, it's about to get tested.

The tempting guess is: the candidate that looks most like the probe wins, and no score can dip below zero, because how could similarity be less than nothing?

Before you run the next cell: the candidates never move below. Only the probe turns. Do you expect the scores to change anyway?

In [ ]:
def probe_at(angle_deg, length=3.0):
    angle = np.radians(angle_deg)
    return np.array([length * np.cos(angle), length * np.sin(angle)])

for angle in [0, 45, 90, 135, 180]:
    p = probe_at(angle)
    row = {name: round(dot_by_hand(vec, p), 2) for name, vec in candidates.items()}
    print(f"angle {angle:>3} deg -> {row}")

The candidates never changed. Only the probe's direction changed, and every score answered. A big score can't mean "same song", because nothing about the songs moved. It means the probe agrees with that direction right now.

And yes, scores go negative. Check it.

In [ ]:
score_facing = dot_by_hand(candidates["A"], probe_at(0))
score_opposite = dot_by_hand(candidates["A"], probe_at(180))

print("facing A:", score_facing)
print("turned away from A:", score_opposite)

assert score_facing > 0
assert score_opposite < 0
assert score_facing != score_opposite

Same candidate, same length, same everything about the song. Only the probe's direction changed, and the score swung from positive to negative. Similarity has a basement.

## Name the machine you just used

Three clean positions tell the whole story: point along a direction, point across it, point against it.

Before you run the next cell: what score do you expect at exactly 90 degrees, across?

In [ ]:
reference = np.array([4.0, 0.0])

along = np.array([2.0, 0.0])     # same direction
across = np.array([0.0, 3.0])    # perpendicular
against = np.array([-2.0, 0.0])  # opposite direction

agree = np.dot(reference, along)
ignore = np.dot(reference, across)
oppose = np.dot(reference, against)

print("agree  (along):", agree)
print("ignore (across):", ignore)
print("oppose (against):", oppose)

assert agree > 0
assert ignore == 0
assert oppose < 0

Agree, positive. Across, exactly zero. Oppose, negative. One multiply-and-add, and it behaves like a meter with a needle swinging from oppose through ignore to agree.

**The dot product is a similarity meter.**

Two lists of numbers go in. One honest reading of alignment comes out. The claim below would be false if the meter's sign didn't track direction alone, so we assert it directly.

In [ ]:
# if the dot product were a similarity meter, its sign must track direction alone,
# not length. Scale "along" way up and it should still read agree.
along_stretched = along * 100

assert np.sign(np.dot(reference, along)) == np.sign(np.dot(reference, along_stretched))
assert np.dot(reference, across) == 0
assert np.sign(np.dot(reference, against)) == -1
print("the meter's sign holds: agree stays positive, across stays zero, oppose stays negative")

## Different aisle, same meter

Swap the songs for a job posting. Score yourself against what the role wants: Python hours, SQL reps, nights free. Multiply the matched entries, add them up, one number for the fit.

The multiply-and-add never asked what the numbers meant. That's exactly why it travels.

In [ ]:
# columns: python_hours, sql_reps, nights_free
you = np.array([20.0, 15.0, 4.0])
role = np.array([10.0, 20.0, 2.0])

fit = dot_by_hand(you, role)
print("job fit score:", fit)

assert fit == np.dot(you, role)
assert fit > 0

## The honest look

A meter you trust blindly is a meter that will lie to you. Take one candidate and stretch it longer, without turning it at all. Its direction should hold dead still. Watch what happens to its score.

Before you run this: do you expect the score to change, even though the direction didn't?

In [ ]:
base = candidates["A"]
stretched = base * 3.0

unit_base = base / np.linalg.norm(base)
unit_stretched = stretched / np.linalg.norm(stretched)

print("base direction:     ", unit_base)
print("stretched direction:", unit_stretched)
print("base score:     ", np.dot(base, probe))
print("stretched score:", np.dot(stretched, probe))

# same direction, unchanged
assert np.allclose(unit_base, unit_stretched)
# but the score climbed anyway
assert np.dot(stretched, probe) > np.dot(base, probe)

Same direction, confirmed by the unit vectors matching. But the score climbed anyway, just because the candidate got longer. A long, loud song can win on sheer size, not on agreement. A padded resume can win on sheer length. The meter can't yet tell "points the same way" apart from "just bigger".

That gap is the next session's opening.

## For the walk home

What is the cheapest fix that keeps the direction and forgets the size?

Rebuild the meter from memory first: two arrows go in, so what two operations turn them into the score? Say it before you scroll back up.

One thing to try next, in code: write a function that divides a vector by its own length before the dot product ever runs, then check whether `stretched` and `base` finally score the same against the probe once you do that. You'll have built cosine similarity before anyone hands you the name.